# Convert Parquet to JSON with chDB

In-process ClickHouse SQL, no server. Run `./generate.sh` first to create `./data/`.
Companion article: https://clickhouse.com/resources/engineering/convert-parquet-to-json

In [ ]:
import os
import chdb

os.chdir("data")

## Convert Parquet -> JSONL (one object per line)
chDB returns the bytes; write them to a file.

In [ ]:
out = chdb.query(
    "SELECT * FROM file('events.parquet') ORDER BY event_id FORMAT JSONEachRow"
)
with open("events_chdb.jsonl", "w") as f:
    f.write(str(out))

with open("events_chdb.jsonl") as f:
    for _ in range(2):
        print(f.readline().rstrip())

{"event_id":1,"event_date":"2026-01-01","event_ts":"2026-01-01 00:00:00.000","country":"GB","amount":5,"tags":["new","paid"],"account":{"name":"acme","tier":1}}
{"event_id":2,"event_date":"2026-01-02","event_ts":"2026-01-01 01:00:00.000","country":"US","amount":6.01,"tags":["web"],"account":{"name":"globex","tier":2}}


## Typed + nested columns carry through
`Array(String)` becomes a JSON array; the named `Tuple` becomes a nested JSON object.

In [ ]:
print(
    chdb.query(
        "SELECT event_id, tags, account FROM file('events.parquet') "
        "ORDER BY event_id LIMIT 2 FORMAT JSONEachRow"
    ),
    end="",
)

{"event_id":1,"tags":["new","paid"],"account":{"name":"acme","tier":1}}
{"event_id":2,"tags":["web"],"account":{"name":"globex","tier":2}}


## Reshape while converting -- it is just SQL

In [ ]:
print(
    chdb.query(
        "SELECT country, count() AS events, round(sum(amount), 2) AS total "
        "FROM file('events.parquet') GROUP BY country ORDER BY total DESC "
        "FORMAT JSONEachRow"
    ),
    end="",
)

{"country":"IN","events":4,"total":66.45}
{"country":"FR","events":4,"total":62.41}
{"country":"DE","events":4,"total":58.38}
{"country":"US","events":4,"total":54.34}
{"country":"GB","events":4,"total":50.29}
